# Generate CO2 Node JSON Snippets for China (24 Basins × 31 Provinces)

Produces four files to paste into `nodes_1.json`:

| File | Contents | Rows |
|------|----------|------|
| `co2_transported_nodes.json` | Intermediate balance nodes (pipeline → injection) | 744 |
| `co2_storage_nodes_min.json` | Storage sinks, saline aquifer **min** injection rate | 24 |
| `co2_storage_nodes_mean.json` | Storage sinks, saline aquifer **mean** injection rate | 24 |
| `co2_storage_nodes_max.json` | Storage sinks, saline aquifer **max** injection rate | 24 |

**Storage constraint unit logic:**  
`rhs_policy` (the `CO2StorageConstraint` budget) is in **tonnes CO2 total** over the modeled period.  
The model represents `NUM_HOURS` out of 8760 hours/year, so:  
`rhs_policy = saline_rate_Mt_yr × 1e6 × (NUM_HOURS / 8760)`

In [9]:
import pandas as pd
import json
import os

NUM_HOURS = 288        # representative hours in the model period
HOURS_PER_YEAR = 8760

BASIN_KEY_MAP = {
    'Songliao Basin':             'Songliao',
    'Tuepan-Hami Basin':          'TurpanHami',   # typo in source CSV
    'Subei Basin':                'Subei',
    'Bohai Bay Basin (onshore)':  'BohaiOnshore',
    'Qaidam Basin':               'Qaidam',
    'Nanxiang Basin':             'Nanxiang',
    'Sanjiang Basin':             'Sanjiang',
    'Hailar Basin':               'Hailar',
    'Jianghan Basin':             'Jianghan',
    'Tarim Basin':                'Tarim',
    'Ordos Basin':                'Ordos',
    'Ejinjina Basin':             'YingenEjina',  # alt transliteration
    'Hehuai Basin':               'Hehuai',
    'Qinshui Basin':              'Qinshui',
    'Erlian Basin':               'Erlian',
    'Junggar Basin':              'Junggar',
    'Sichuan Basin':              'SichuanBasin',
    'Bohai Bay Basin (offshore)': 'BohaiOffshore',
    'North Yellow Sea Basin':     'NorthYellowSea',
    'South Yellow Sea Basin':     'SouthYellowSea',
    'East China Sea Basin':       'EastChinaSea',
    'Pearl River Mouth Basin':    'PearlRiverMouth',
    'Beibu Gulf Basin':           'BeibugGulf',
    'Qiongdongnan Basin':         'Qiongdongnan',
}

SALINE_COLS = {
    'min':  'Saline Aquifer Min (Mt/a)',
    'mean': 'Saline Aquifer Mean (Mt/a)',
    'max':  'Saline Aquifer Max (Mt/a)',
}

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('make_co2_storage_nodes.ipynb'))
INJECTION_CSV  = os.path.join(NOTEBOOK_DIR, 'nature_scientific_data_source', 'injection_rates.csv')
PIPELINE_CSV   = os.path.join(NOTEBOOK_DIR, 'co2_pipeline.csv')

rates_df = pd.read_csv(INJECTION_CSV)
for col in SALINE_COLS.values():
    rates_df[col] = pd.to_numeric(rates_df[col], errors='coerce').fillna(0.0)
rates_df['basin_key'] = rates_df['Basin'].map(BASIN_KEY_MAP)
unmatched = rates_df[rates_df['basin_key'].isna()]['Basin'].tolist()
if unmatched:
    print(f'WARNING: unmatched basin names: {unmatched}')
rates_df = rates_df[rates_df['basin_key'].notna()].copy()
print(f'Loaded {len(rates_df)} basins from injection rates CSV')
print(f'Model period: {NUM_HOURS} hrs  ({NUM_HOURS/HOURS_PER_YEAR*100:.2f}% of year)')

Loaded 24 basins from injection rates CSV
Model period: 288 hrs  (3.29% of year)


In [10]:
# ── 1. co2_transported nodes (744 intermediate balance nodes) ─────────────────
pipeline_df = pd.read_csv(PIPELINE_CSV)
transported_ids = sorted(pipeline_df['edges--transmission_edge--end_vertex'].unique())

transported_section = {
    "type": "CO2Captured",
    "global_data": {
        "time_interval": "CO2Captured",
        "constraints": {"BalanceConstraint": True}
    },
    "instance_data": [{"id": vtx_id} for vtx_id in transported_ids]
}

out_path = os.path.join(NOTEBOOK_DIR, 'co2_transported_nodes.json')
with open(out_path, 'w') as f:
    json.dump(transported_section, f, indent=2)

print(f'co2_transported_nodes.json  ->  {len(transported_ids)} nodes')
print('First 3:', transported_ids[:3])
print('Last 3: ', transported_ids[-3:])

co2_transported_nodes.json  ->  744 nodes
First 3: ['co2_transported_Region10Jiangsu_to_BeibugGulf', 'co2_transported_Region10Jiangsu_to_BohaiOffshore', 'co2_transported_Region10Jiangsu_to_BohaiOnshore']
Last 3:  ['co2_transported_Region9Shanghai_to_Tarim', 'co2_transported_Region9Shanghai_to_TurpanHami', 'co2_transported_Region9Shanghai_to_YingenEjina']


In [11]:
# ── 2. co2_storage nodes (24 sinks, one file per estimate) ───────────────────
def make_storage_section(df, saline_col):
    instance_data = []
    for _, row in df.iterrows():
        rhs = int(round(row[saline_col] * 1e6 * NUM_HOURS / HOURS_PER_YEAR))
        instance_data.append({
            "id": f"co2_storage_{row['basin_key']}",
            "constraints": {"CO2StorageConstraint": True},
            "rhs_policy": {"CO2StorageConstraint": rhs}
        })
    return {
        "type": "CO2Captured",
        "global_data": {
            "time_interval": "CO2Captured",
            "constraints": {"BalanceConstraint": False}
        },
        "instance_data": instance_data
    }

storage_files = {}
for scenario, col in SALINE_COLS.items():
    section = make_storage_section(rates_df, col)
    fname = f'co2_storage_nodes_{scenario}.json'
    out_path = os.path.join(NOTEBOOK_DIR, fname)
    with open(out_path, 'w') as f:
        json.dump(section, f, indent=2)
    storage_files[scenario] = out_path
    rhs_vals = [x['rhs_policy']['CO2StorageConstraint'] for x in section['instance_data']]
    print(f'{scenario:4s}  ->  {fname}')
    print(f'        rhs range: {min(rhs_vals):>10,} – {max(rhs_vals):>10,} t over {NUM_HOURS} hrs')
    print(f'        total saline: {rates_df[col].sum():.1f} Mt/yr')
    print()

min   ->  co2_storage_nodes_min.json
        rhs range:         99 –    262,685 t over 288 hrs
        total saline: 35.3 Mt/yr

mean  ->  co2_storage_nodes_mean.json
        rhs range:        329 –  1,645,479 t over 288 hrs
        total saline: 259.8 Mt/yr

max   ->  co2_storage_nodes_max.json
        rhs range:        658 –  4,418,630 t over 288 hrs
        total saline: 1030.8 Mt/yr



In [12]:
# ── Cross-checks ─────────────────────────────────────────────────────────────
inj_csv = os.path.join(NOTEBOOK_DIR, 'co2_injection.csv')
inj_df  = pd.read_csv(inj_csv)

# transported nodes: pipeline end_vertex == injection start_vertex == transported node IDs
pipe_ends  = set(pipeline_df['edges--transmission_edge--end_vertex'])
inj_starts = set(inj_df['edges--co2_captured_edge--start_vertex'])
node_set   = set(transported_ids)
assert pipe_ends == inj_starts == node_set, 'Transported vertex mismatch'
print(f'Transported nodes: {len(node_set)} — consistent across pipeline, injection, and node file.')

# storage nodes: injection end_vertex == storage node IDs
storage_vtx = set(inj_df['edges--co2_storage_edge--end_vertex'])
storage_ids = {f"co2_storage_{r['basin_key']}" for _, r in rates_df.iterrows()}
assert storage_vtx == storage_ids, f'Storage vertex mismatch: {storage_vtx.symmetric_difference(storage_ids)}'
print(f'Storage nodes:     {len(storage_ids)} — consistent across injection and node files.')
print('\nAll checks passed.')

Transported nodes: 744 — consistent across pipeline, injection, and node file.
Storage nodes:     24 — consistent across injection and node files.

All checks passed.
